# 🛒 AluraStoreBr – Análise de Desempenho das Lojas

> **Desafio Data Science** | Auxiliar o Senhor João a decidir qual loja da rede Alura Store deve ser vendida.

---

## 1. Importação das Bibliotecas e Carregamento dos Dados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Carrega os CSVs locais
lojas = {
    'Loja 1': pd.read_csv('loja_1.csv'),
    'Loja 2': pd.read_csv('loja_2.csv'),
    'Loja 3': pd.read_csv('loja_3.csv'),
    'Loja 4': pd.read_csv('loja_4.csv'),
}

CORES = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
NOMES = list(lojas.keys())

for nome, df in lojas.items():
    print(f'{nome}: {len(df)} registros | Colunas: {list(df.columns)}')

## 2. Faturamento Total por Loja

In [ ]:
faturamento = {nome: df['Preço'].sum() for nome, df in lojas.items()}

print('Faturamento Total por Loja')
print('-' * 35)
for nome, val in faturamento.items():
    print(f'{nome}: R$ {val:,.2f}')

## 3. Vendas por Categoria

In [ ]:
for nome, df in lojas.items():
    print(f'\n{nome}')
    print(df['Categoria do Produto'].value_counts().to_string())

## 4. Média de Avaliação por Loja

In [ ]:
avaliacoes = {nome: df['Avaliação da compra'].mean() for nome, df in lojas.items()}

print('Média de Avaliação')
print('-' * 30)
for nome, media in avaliacoes.items():
    print(f'{nome}: {media:.2f} ⭐')

## 5. Produtos Mais e Menos Vendidos

In [ ]:
for nome, df in lojas.items():
    contagem = df['Produto'].value_counts()
    print(f'\n{nome}')
    print(f'  Mais vendido : {contagem.index[0]} ({contagem.iloc[0]}x)')
    print(f'  Menos vendido: {contagem.index[-1]} ({contagem.iloc[-1]}x)')

## 6. Frete Médio por Loja

In [ ]:
fretes = {nome: df['Frete'].mean() for nome, df in lojas.items()}

print('Frete Médio')
print('-' * 30)
for nome, frete in fretes.items():
    print(f'{nome}: R$ {frete:.2f}')

## 7. Visualizações

### 7.1 Faturamento Total (Gráfico de Barras)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(NOMES, [faturamento[n] for n in NOMES], color=CORES, edgecolor='white', width=0.55)
for bar, val in zip(bars, faturamento.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+15000,
            f'R$ {val/1e6:.2f}M', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Faturamento Total por Loja', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Faturamento (R$)', fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R$ {x/1e6:.1f}M'))
ax.set_ylim(0, max(faturamento.values()) * 1.15)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('grafico_faturamento.png', dpi=150)
plt.show()

### 7.2 Média de Avaliação (Barras Horizontais)

In [ ]:
medias = [avaliacoes[n] for n in NOMES]
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(NOMES, medias, color=CORES, edgecolor='white', height=0.5)
for bar, val in zip(bars, medias):
    ax.text(val+0.02, bar.get_y()+bar.get_height()/2,
            f'{val:.2f} ⭐', va='center', fontsize=11, fontweight='bold')
ax.set_xlim(0, 5.5)
ax.set_title('Média de Avaliação por Loja', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Avaliação Média (1–5)', fontsize=11)
ax.axvline(x=sum(medias)/len(medias), color='gray', linestyle='--', alpha=0.6, label='Média geral')
ax.legend(); ax.grid(axis='x', linestyle='--', alpha=0.4)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('grafico_avaliacoes.png', dpi=150)
plt.show()

### 7.3 Vendas por Categoria (Barras Agrupadas)

In [ ]:
todas_categorias = set()
for df in lojas.values(): todas_categorias.update(df['Categoria do Produto'].unique())
cat_data = {nome: {cat: df['Categoria do Produto'].value_counts().get(cat,0) for cat in todas_categorias} for nome, df in lojas.items()}
cat_df = pd.DataFrame(cat_data).T
cat_df = cat_df[cat_df.sum().sort_values(ascending=False).index]
fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(NOMES))
n_cats = len(cat_df.columns)
width = 0.8 / n_cats
palette = plt.cm.tab10.colors
for i, cat in enumerate(cat_df.columns):
    offsets = [xi + i*width - (n_cats-1)*width/2 for xi in x]
    ax.bar(offsets, cat_df[cat], width=width*0.9, label=cat, color=palette[i%10])
ax.set_xticks(list(x)); ax.set_xticklabels(NOMES, fontsize=11)
ax.set_title('Vendas por Categoria em Cada Loja', fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel('Quantidade de Vendas', fontsize=11)
ax.legend(title='Categoria', bbox_to_anchor=(1.01,1), loc='upper left', fontsize=8)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('grafico_categorias.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.4 Frete Médio (Gráfico de Pizza)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
valores_frete = [fretes[n] for n in NOMES]
wedges, texts, autotexts = ax.pie(valores_frete, labels=NOMES, colors=CORES,
    autopct=lambda p: f'R$ {p/100*sum(valores_frete):.2f}',
    startangle=140, explode=[0.05]*4,
    wedgeprops=dict(edgecolor='white', linewidth=1.5))
for at in autotexts: at.set_fontsize(10); at.set_fontweight('bold')
ax.set_title('Distribuição do Frete Médio por Loja', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('grafico_frete.png', dpi=150)
plt.show()

### 7.5 Dispersão: Faturamento × Avaliação

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for i, nome in enumerate(NOMES):
    ax.scatter(faturamento[nome], avaliacoes[nome], color=CORES[i], s=250, zorder=5, edgecolors='white', linewidth=1.5)
    ax.annotate(nome, xy=(faturamento[nome], avaliacoes[nome]), xytext=(8,5), textcoords='offset points', fontsize=10, fontweight='bold')
ax.set_xlabel('Faturamento Total (R$)', fontsize=11)
ax.set_ylabel('Média de Avaliação', fontsize=11)
ax.set_title('Faturamento vs. Avaliação por Loja', fontsize=14, fontweight='bold', pad=15)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'R$ {x/1e6:.2f}M'))
ax.grid(linestyle='--', alpha=0.4); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('grafico_dispersao.png', dpi=150)
plt.show()

## 8. Recomendação Final

Com base na análise dos dados das quatro lojas da Alura Store, a recomendação é que o **Senhor João venda a Loja 4**.

### Justificativa:

1. **Menor Faturamento** – A Loja 4 registrou o menor faturamento total (R$ 1.384.497,58), cerca de **R$ 150.000 a menos** que a Loja 1 (melhor colocada). Isso indica menor geração de receita e menor atratividade comercial.

2. **Avaliação dos Clientes** – Embora todas as lojas tenham avaliação próxima, a Loja 4 apresenta a segunda pior média (4,00 ⭐), ficando abaixo das Lojas 2 e 3.

3. **Ticket Médio e Mix de Produtos** – O faturamento mais baixo combinado com volume de vendas semelhante ao das outras lojas indica um ticket médio menor, ou seja, os produtos mais vendidos na Loja 4 têm preço inferior.

4. **Desempenho Consolidado** – Ao cruzar faturamento e avaliação no gráfico de dispersão, a Loja 4 ocupa claramente a posição de menor eficiência geral.

### Conclusão:

> A **Loja 4** é a unidade com o pior desempenho consolidado da rede Alura Store. Vendê-la permitirá ao Senhor João concentrar recursos nas demais unidades ou investir com mais segurança em um novo empreendimento com maior potencial de retorno.